# Inspect PMID Evidence for a Pathogen-Antibiotic AMR Pair

Use this notebook when a pathogen-antibiotic resistance association looks suspicious and you want to inspect the accepted PubMed evidence by year.


In [ ]:
from pathlib import Path
import sys

from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipelines import build_pair_pmid_title_table, pair_pmid_table_html, save_pair_pmid_excel, slugify_output_name
from amr.pipeline_config import cfg, cfg_path, load_pipeline_config


## Parameters


In [ ]:
PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)

OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.output_dir", PROJECT_ROOT)
PARQUET_OUTPUT_DIR = cfg_path(PIPELINE_CONFIG, "paths.parquet_dir", PROJECT_ROOT)

YEARLY_SCREENED_PATH = cfg_path(PIPELINE_CONFIG, "paths.yearly_screened", PROJECT_ROOT)
ABSTRACT_CACHE_PATH = cfg_path(PIPELINE_CONFIG, "paths.abstract_cache", PROJECT_ROOT)

# Pair displayed by this inspection notebook.
PAIR_PATHOGEN = cfg(PIPELINE_CONFIG, "pair_pmids.pathogen")
PAIR_ANTIBIOTIC = cfg(PIPELINE_CONFIG, "pair_pmids.antibiotic")
MAX_PMIDS_PER_YEAR = cfg(PIPELINE_CONFIG, "pair_pmids.max_pmids_per_year")

PAIR_PMID_EXCEL_PATH = (
    OUTPUT_DIR
    / f"pair_pmids__{slugify_output_name(PAIR_PATHOGEN)}__{slugify_output_name(PAIR_ANTIBIOTIC)}.xlsx"
)


## PMID Evidence Table


In [ ]:
pair_pmid_df = build_pair_pmid_title_table(
    YEARLY_SCREENED_PATH,
    ABSTRACT_CACHE_PATH,
    pathogen=PAIR_PATHOGEN,
    antibiotic=PAIR_ANTIBIOTIC,
    max_pmids_per_year=MAX_PMIDS_PER_YEAR,
)

print(
    f"Found {len(pair_pmid_df):,} displayed PMID rows for "
    f"{PAIR_PATHOGEN} vs {PAIR_ANTIBIOTIC} "
    f"with up to {MAX_PMIDS_PER_YEAR:,} PMIDs per year."
)
display(HTML(pair_pmid_table_html(pair_pmid_df)))

saved_excel_path = save_pair_pmid_excel(pair_pmid_df, PAIR_PMID_EXCEL_PATH)
print(f"Saved Excel file -> {saved_excel_path}")


In [ ]:
display(HTML(pair_pmid_table_html(pair_pmid_df)))